<a href="https://colab.research.google.com/github/Rushikeshaya/Generative-AI/blob/main/%22Fresh_and_Rotten_Fruits_Classification_Using_CNN_and_Transfer_Learning%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
import tensorflow as tf
tf.keras.mixed_precision.set_global_policy('mixed_float16')
print("Mixed precision enabled with 'mixed_float16' policy.")

Mixed precision enabled with 'mixed_float16' policy.


##Library Imports

# Google Colab Lab Assignment -Pretrained Modle

**Course Name:** Generative AI

**Lab Title:** "Fresh and Rotten Fruits Classification Using CNN and Transfer Learning"

**Student Name:** Yadnesh Tayade

**Student ID:** 202401110057

**Date of Submission:** 20-08-26

**Group Members**: Rushikesh Hande , Soham sabane

**PRN** 202401110056 , 202401110070

**Research Paper Study and Implementation**

**Instructions:**

1. Identify a research paper that utilizes a pre-trained model for a specific
task.

2. Study the methodology, dataset, and model used in the research paper.

3. Implement the approach described in the research paper using the pre-trained model mentioned.

4. Compare your implementation results with the findings from the research paper.

**Objective**
1.   Study a research paper utilizing a pre-trained model.
2.   Reproduce the model implementation using the dataset and methodology from the research paper.
3.   Fine-tune the pre-trained model and optimize hyperparameters.
3.   Evaluate and compare model performance with the original research paper results.









**Task 1: Research Paper Selection and Dataset Preparation (2 hours)**

**Instructions:**

1. Select a research paper that applies a pre-trained model (e.g., VGG, ResNet, EfficientNet, etc.).

2. Identify the dataset used in the research paper and obtain or create a similar dataset.(**Mention Dataset Link and Description**)

3. Perform necessary preprocessing steps:

 Resize images to match the model input dimensions.

 Apply data augmentation techniques if applicable.

4. Split the dataset into training, validation, and testing sets.

**Research Paper:** Image Classification for Snow Detection to Improve Pedestrian Safety\
**Research Paper Link:** https://www.ijrar.org/papers/IJRAR22C1060.pdf \
**Dataset Link:** https://www.kaggle.com/datasets/sriramr/fruits-fresh-and-rotten-for-classification

In [11]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.optimizers import Adam
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


##Dataset Configuration & Preprocessing

In [12]:
import numpy as np
import tensorflow as tf

# Load CIFAR-10 dataset
print("Loading CIFAR-10 dataset...")
(cifar_x_train, cifar_y_train), (cifar_x_test, cifar_y_test) = tf.keras.datasets.cifar10.load_data()
print(f"CIFAR-10 original train samples: {cifar_x_train.shape[0]}, test samples: {cifar_x_test.shape[0]}")

# Define the number of samples you want per class for your tiny dataset
# CIFAR-10 has 10 classes. If you want ~50-60 images, 5-6 images per class is ideal.
num_samples_per_class_train = 5 # 5 images per class for training
num_samples_per_class_test = 1  # 1 image per class for testing

def create_fixed_size_subset(images, labels, num_samples_per_class):
    subset_images = []
    subset_labels = []
    # Ensure labels are 1D
    labels_flat = labels.flatten()
    unique_classes = np.unique(labels_flat)

    for class_id in unique_classes:
        class_indices = np.where(labels_flat == class_id)[0]
        # Sample without replacement; ensure we don't ask for more than available
        num_to_sample = min(num_samples_per_class, len(class_indices))
        sampled_indices = np.random.choice(class_indices, num_to_sample, replace=False)

        subset_images.append(images[sampled_indices])
        subset_labels.append(labels[sampled_indices])

    return np.concatenate(subset_images, axis=0), np.concatenate(subset_labels, axis=0)

# Create tiny subsets
tiny_x_train, tiny_y_train = create_fixed_size_subset(cifar_x_train, cifar_y_train, num_samples_per_class_train)
tiny_x_test, tiny_y_test = create_fixed_size_subset(cifar_x_test, cifar_y_test, num_samples_per_class_test)

print(f"Created Tiny Training Subset: {tiny_x_train.shape[0]} samples") # Should be 10 classes * 5 = 50 samples
print(f"Created Tiny Test Subset: {tiny_x_test.shape[0]} samples") # Should be 10 classes * 1 = 10 samples

# Function to prepare this tiny numpy array data into a tf.data.Dataset
def prepare_tiny_tf_dataset(images, labels, batch_size, img_size, shuffle=True):
    ds = tf.data.Dataset.from_tensor_slices((images, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=100) # Small buffer for tiny dataset

    # Preprocess: resize to IMG_SIZE and normalize to [0, 1]
    ds = ds.map(lambda x, y: (tf.image.resize(tf.cast(x, tf.float32) / 255.0, img_size), y),
               num_parallel_calls=tf.data.AUTOTUNE)

    ds = ds.batch(batch_size)
    ds = ds.prefetch(buffer_size=tf.data.AUTOTUNE)
    return ds

# Prepare the tiny datasets using your existing BATCH_SIZE and IMG_SIZE
tiny_train_ds = prepare_tiny_tf_dataset(tiny_x_train, tiny_y_train, BATCH_SIZE, IMG_SIZE)
tiny_test_ds = prepare_tiny_tf_dataset(tiny_x_test, tiny_y_test, BATCH_SIZE, IMG_SIZE, shuffle=False)

print("\nTiny datasets prepared for training and testing.")
print("You can now use `tiny_train_ds` and `tiny_test_ds` in `model.fit()` for very fast runs.")

# Example of how you'd use it:
# history_cnn_tiny = custom_cnn.fit(
#     tiny_train_ds,
#     validation_data=tiny_test_ds,
#     epochs=EPOCHS # Maybe reduce epochs for tiny dataset too, e.g., epochs=1
# )

Loading CIFAR-10 dataset...
170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 1349s 8us/step
CIFAR-10 original train samples: 50000, test samples: 10000
Created Tiny Training Subset: 50 samples
Created Tiny Test Subset: 10 samples

Tiny datasets prepared for training and testing.
You can now use `tiny_train_ds` and `tiny_test_ds` in `model.fit()` for very fast runs.


In [13]:
# Configurations
IMG_SIZE = (64, 64)
BATCH_SIZE = 64
EPOCHS = 10
TRAIN_DIR = "/content/drive/MyDrive/dataset/train"
TEST_DIR = "/content/drive/MyDrive/dataset/test"

# Load training and test datasets
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True
)

test_ds = tf.keras.preprocessing.image_dataset_from_directory(
    TEST_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False  # Crucial for matching predictions to true labels
)

class_names = train_ds.class_names
num_classes = len(class_names)
print(f"Detected {num_classes} classes: {class_names}")

# Performance optimization: prefetch & cache
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.cache().prefetch(buffer_size=AUTOTUNE)

# Extract ground-truth test labels once for evaluation
y_true = np.concatenate([y.numpy() for _, y in test_ds], axis=0)

Found 10337 files belonging to 6 classes.
Found 2712 files belonging to 6 classes.
Detected 6 classes: ['freshapples', 'freshbanana', 'freshoranges', 'rottenapples', 'rottenbanana', 'rottenoranges']


### Option: Creating a Very Small Dataset from Your Own Mounted Data

To ensure immediate execution and avoid any further downloads, we will create an extremely small subset directly from your already loaded `train_ds` and `test_ds`. This will allow you to run your models for a few epochs very quickly to check your pipeline.

In [14]:
# Create tiny subsets from your existing loaded datasets
# We'll take a very small number of batches for ultra-fast experimentation.

# Number of batches to take for the tiny training dataset
# Adjust this value to control the size of your tiny dataset
num_train_batches_for_tiny_ds = 2 # e.g., 2 batches (2 * BATCH_SIZE samples)
num_test_batches_for_tiny_ds = 1  # e.g., 1 batch (1 * BATCH_SIZE samples)

print(f"Creating tiny training dataset by taking {num_train_batches_for_tiny_ds} batches from train_ds...")
tiny_train_ds = train_ds.take(num_train_batches_for_tiny_ds)

print(f"Creating tiny test dataset by taking {num_test_batches_for_tiny_ds} batches from test_ds...")
tiny_test_ds = test_ds.take(num_test_batches_for_tiny_ds)

print(f"\nTiny datasets created. Each epoch will now process approximately "
      f"{num_train_batches_for_tiny_ds * BATCH_SIZE} training samples and "
      f"{num_test_batches_for_tiny_ds * BATCH_SIZE} validation samples.")

# IMPORTANT: When you want to train on the full dataset again, revert these changes
# or re-run the cell above this to reset `train_ds` and `test_ds` to their full versions.

Creating tiny training dataset by taking 2 batches from train_ds...
Creating tiny test dataset by taking 1 batches from test_ds...

Tiny datasets created. Each epoch will now process approximately 128 training samples and 64 validation samples.


**Task 2: Model Implementation and Fine-tuning**

**Instructions:**

1. Implement the pre-trained model as described in the research paper.

2. Visualize feature maps of few layers

3. Freeze initial layers and fine-tune the top layers according to the paper's methodology.

4. Optimize hyperparameters such as:

  Learning rate

  Batch size

  Number of epochs

  Optimizer choice (Adam, SGD, RMSprop, etc.)

4. Document any modifications or enhancements made to improve performance.

##Custom CNN Architecture

### Custom CNN: Definition, Compilation, and Training (for quick test)

In [ ]:
import time
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam

def build_custom_cnn(input_shape=(128, 128, 3), num_classes=6):
    model = models.Sequential([
        # Data Rescaling / Normalization [0, 255] -> [0, 1]
        layers.Rescaling(1.0 / 255, input_shape=input_shape),

        # Conv Block 1
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),

        # Conv Block 2
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),

        # Conv Block 3
        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),

        # Dense Classifier Head
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ])
    return model

print("Building and compiling Custom CNN...")
custom_cnn = build_custom_cnn(input_shape=(*IMG_SIZE, 3), num_classes=num_classes)
custom_cnn.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
custom_cnn.summary()

print("\n--- Training Custom CNN (with Tiny Dataset) ---")
start_time_cnn = time.time()
history_cnn = custom_cnn.fit(
    tiny_train_ds,
    validation_data=tiny_test_ds,
    epochs=1 # Set to 1 for very quick test
)
training_time_cnn = time.time() - start_time_cnn
print(f"\n[Custom CNN] Training Time (Tiny Dataset): {training_time_cnn:.2f} seconds")

Building and compiling Custom CNN...


/usr/local/lib/python3.13/dist-packages/keras/src/layers/preprocessing/data_layer.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling_1 (Rescaling)         │ (None, 64, 64, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 64, 64, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 64, 64, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 32, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 32, 32, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 32, 32, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 16, 16, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 16, 16, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 8, 8, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 8192)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │     1,048,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,143,622 (4.36 MB)

 Trainable params: 1,143,174 (4.36 MB)

 Non-trainable params: 448 (1.75 KB)


--- Training Custom CNN (with Tiny Dataset) ---


### Transfer Learning Model (MobileNetV2): Definition, Compilation, and Training (for quick test)

In [ ]:
import time
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.optimizers import Adam

def build_transfer_learning_model(input_shape=(128, 128, 3), num_classes=6):
    # Pre-trained base model
    base_model = MobileNetV2(
        weights='imagenet',
        include_top=False,
        input_shape=input_shape
    )
    # Freeze base model layers for feature extraction
    base_model.trainable = False

    inputs = tf.keras.Input(shape=input_shape)
    # MobileNetV2 expects pixel scaling to [-1, 1]
    x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    return models.Model(inputs, outputs), base_model

print("Building and compiling Transfer Learning Model (MobileNetV2)...")
tl_model, base_mobilenet = build_transfer_learning_model(input_shape=(*IMG_SIZE, 3), num_classes=num_classes)
tl_model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
tl_model.summary()

print("\n--- Training Transfer Learning Model (MobileNetV2 with Tiny Dataset) ---")
start_time_tl = time.time()
history_tl = tl_model.fit(
    tiny_train_ds,
    validation_data=tiny_test_ds,
    epochs=1 # Set to 1 for very quick test
)
training_time_tl = time.time() - start_time_tl
print(f"\n[Transfer Learning] Training Time (Tiny Dataset): {training_time_tl:.2f} seconds")

In [ ]:
def build_custom_cnn(input_shape=(128, 128, 3), num_classes=6):
    model = models.Sequential([
        # Data Rescaling / Normalization [0, 255] -> [0, 1]
        layers.Rescaling(1.0 / 255, input_shape=input_shape),

        # Conv Block 1
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),

        # Conv Block 2
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),

        # Conv Block 3
        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),

        # Dense Classifier Head
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ])
    return model

custom_cnn = build_custom_cnn(input_shape=(*IMG_SIZE, 3), num_classes=num_classes)
custom_cnn.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
custom_cnn.summary()

##Train Custom CNN

In [ ]:
print("\n--- Training Custom CNN (with Tiny Dataset) ---")
start_time_cnn = time.time()
history_cnn = custom_cnn.fit(
    tiny_train_ds,
    validation_data=tiny_test_ds,
    epochs=1 # Set to 1 for very quick test
)
training_time_cnn = time.time() - start_time_cnn
print(f"\n[Custom CNN] Training Time (Tiny Dataset): {training_time_cnn:.2f} seconds")

##Transfer Learning (MobileNetV2) Architecture

In [ ]:
def build_transfer_learning_model(input_shape=(128, 128, 3), num_classes=6):
    # Pre-trained base model
    base_model = MobileNetV2(
        weights='imagenet',
        include_top=False,
        input_shape=input_shape
    )
    # Freeze base model layers for feature extraction
    base_model.trainable = False

    inputs = tf.keras.Input(shape=input_shape)
    # MobileNetV2 expects pixel scaling to [-1, 1]
    x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    return models.Model(inputs, outputs), base_model

tl_model, base_mobilenet = build_transfer_learning_model(input_shape=(*IMG_SIZE, 3), num_classes=num_classes)
tl_model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
tl_model.summary()

##Train Transfer Learning Model

In [ ]:
print("\n--- Training Transfer Learning Model (MobileNetV2 with Tiny Dataset) ---")
start_time_tl = time.time()
history_tl = tl_model.fit(
    tiny_train_ds,
    validation_data=tiny_test_ds,
    epochs=1 # Set to 1 for very quick test
)
training_time_tl = time.time() - start_time_tl
print(f"\n[Transfer Learning] Training Time (Tiny Dataset): {training_time_tl:.2f} seconds")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

print("Step 1/3: Generating Predictions...")
y_pred_probs_cnn = custom_cnn.predict(test_ds)
y_pred_cnn = np.argmax(y_pred_probs_cnn, axis=1)
report_cnn = classification_report(y_true, y_pred_cnn, target_names=class_names, output_dict=True)

y_pred_probs_tl = tl_model.predict(test_ds)
y_pred_tl = np.argmax(y_pred_probs_tl, axis=1)
report_tl = classification_report(y_true, y_pred_tl, target_names=class_names, output_dict=True)

print("\nStep 2/3: Generating Comparative Summary Table...")
comparison_data = {
    "Metric": ["Total Params", "Accuracy", "Precision (Macro)", "Recall (Macro)", "F1-Score (Macro)"],
    "Custom CNN": [
        f"{custom_cnn.count_params():,}",
        f"{report_cnn['accuracy']*100:.2f}%",
        f"{report_cnn['macro avg']['precision']*100:.2f}%",
        f"{report_cnn['macro avg']['recall']*100:.2f}%",
        f"{report_cnn['macro avg']['f1-score']*100:.2f}%"
    ],
    "MobileNetV2": [
        f"{tl_model.count_params():,}",
        f"{report_tl['accuracy']*100:.2f}%",
        f"{report_tl['macro avg']['precision']*100:.2f}%",
        f"{report_tl['macro avg']['recall']*100:.2f}%",
        f"{report_tl['macro avg']['f1-score']*100:.2f}%"
    ]
}
display(pd.DataFrame(comparison_data))

print("\nStep 3/3: Plotting Visualizations...")
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
cm_cnn = confusion_matrix(y_true, y_pred_cnn)
sns.heatmap(cm_cnn, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names, ax=axes[0])
axes[0].set_title('Custom CNN CM')

cm_tl = confusion_matrix(y_true, y_pred_tl)
sns.heatmap(cm_tl, annot=True, fmt='d', cmap='Greens', xticklabels=class_names, yticklabels=class_names, ax=axes[1])
axes[1].set_title('MobileNetV2 CM')
plt.show()

##Predictions & Classification Reports

In [ ]:
# Efficient vectorized predictions
y_pred_probs_cnn = custom_cnn.predict(test_ds)
y_pred_cnn = np.argmax(y_pred_probs_cnn, axis=1)
report_cnn = classification_report(y_true, y_pred_cnn, target_names=class_names, output_dict=True)

y_pred_probs_tl = tl_model.predict(test_ds)
y_pred_tl = np.argmax(y_pred_probs_tl, axis=1)
report_tl = classification_report(y_true, y_pred_tl, target_names=class_names, output_dict=True)

def predict_single_image(model, image_path):
    """Utility to predict class and confidence for a single image file."""
    img = tf.keras.preprocessing.image.load_img(image_path, target_size=IMG_SIZE)
    img_array = tf.keras.preprocessing.image.img_to_array(img)
    img_batch = np.expand_dims(img_array, axis=0)

    predictions = model.predict(img_batch, verbose=0)
    predicted_class = class_names[np.argmax(predictions[0])]
    confidence = np.max(predictions[0]) * 100
    print(f"Image: {image_path} -> Predicted: {predicted_class} ({confidence:.2f}%)")

##Comparative Study Summary Table

In [ ]:
comparison_data = {
    "Evaluation Parameter": [
        "Total Parameters",
        "Trainable Parameters",
        "Training Time (s)",
        "Overall Accuracy",
        "Macro Avg Precision",
        "Macro Avg Recall",
        "Macro Avg F1-Score"
    ],
    "Custom CNN": [
        f"{custom_cnn.count_params():,}",
        f"{sum([tf.size(w).numpy() for w in custom_cnn.trainable_weights]):,}",
        f"{training_time_cnn:.2f}s",
        f"{report_cnn['accuracy'] * 100:.2f}%",
        f"{report_cnn['macro avg']['precision'] * 100:.2f}%",
        f"{report_cnn['macro avg']['recall'] * 100:.2f}%",
        f"{report_cnn['macro avg']['f1-score'] * 100:.2f}%"
    ],
    "Transfer Learning (MobileNetV2)": [
        f"{tl_model.count_params():,}",
        f"{sum([tf.size(w).numpy() for w in tl_model.trainable_weights]):,}",
        f"{training_time_tl:.2f}s",
        f"{report_tl['accuracy'] * 100:.2f}%",
        f"{report_tl['macro avg']['precision'] * 100:.2f}%",
        f"{report_tl['macro avg']['recall'] * 100:.2f}%",
        f"{report_tl['macro avg']['f1-score'] * 100:.2f}%"
    ]
}

comparison_df = pd.DataFrame(comparison_data)
print("\n" + "=" * 70)
print("                   COMPARATIVE STUDY SUMMARY")
print("=" * 70)
print(comparison_df.to_string(index=False))

#**Task 3: Model Evaluation and Performance Comparison**

**Instructions:**

1. Evaluate the trained model using performance metrics:

 Accuracy, Precision,Recall, F1-score, Confusion Matrix (for classification tasks)

2. Compare the results with those reported in the research paper.

3. Identify potential weaknesses and suggest improvements.
**Deliverables:**

Performance metrics summary (table or chart).

Graphs/plots showcasing model accuracy and loss trends.

Comparison with research paper results.

Discussion on model performance and areas for improvement.

##Visualizations (Curves & Confusion Matrices)

In [ ]:
# Plot Accuracy and Loss Curves
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(history_cnn.history['accuracy'], label='CNN Train Acc', linestyle='--')
plt.plot(history_cnn.history['val_accuracy'], label='CNN Val Acc')
plt.plot(history_tl.history['accuracy'], label='MobileNetV2 Train Acc', linestyle='--')
plt.plot(history_tl.history['val_accuracy'], label='MobileNetV2 Val Acc')
plt.title('Model Accuracy Comparison')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history_cnn.history['loss'], label='CNN Train Loss', linestyle='--')
plt.plot(history_cnn.history['val_loss'], label='CNN Val Loss')
plt.plot(history_tl.history['loss'], label='MobileNetV2 Train Loss', linestyle='--')
plt.plot(history_tl.history['val_loss'], label='MobileNetV2 Val Loss')
plt.title('Model Loss Comparison')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

# Plot Confusion Matrices
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm_cnn = confusion_matrix(y_true, y_pred_cnn)
sns.heatmap(cm_cnn, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names, ax=axes[0])
axes[0].set_title('Custom CNN Confusion Matrix')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('True')

cm_tl = confusion_matrix(y_true, y_pred_tl)
sns.heatmap(cm_tl, annot=True, fmt='d', cmap='Greens', xticklabels=class_names, yticklabels=class_names, ax=axes[1])
axes[1].set_title('MobileNetV2 Confusion Matrix')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('True')

plt.tight_layout()
plt.show()

## Discussion on Model Performance & Areas for Improvement

### Model Weaknesses
* **Small Dataset & Overfitting:** The scratch-built custom CNN struggles to generalize to unseen data due to the limited size of the dataset, making it highly prone to overfitting.
* **Limited Feature Adaptation:** Because the MobileNetV2 base layers were completely frozen during transfer learning, the model relies strictly on generic pre-trained features and cannot adapt its deeper weights to highly specific domain patterns.
* **Visual Ambiguity:** Varying lighting conditions, shadows, and subtle visual similarities between classes can easily confuse the models and lead to misclassifications.

### Proposed Enhancements
1. **Dataset Expansion:** Introduce a much larger and more diverse dataset capturing different lighting, angles, and environmental conditions to boost model generalization.
2. **Data Augmentation & Fine-Tuning:** Implement augmentation techniques (rotation, flipping, contrast adjustments) and unfreeze the top layers of the MobileNetV2 backbone to fine-tune the feature extractor specifically for this classification task.
3. **Hyperparameter Optimization:** Experiment with different learning rates, alternative optimizers (like SGD with momentum instead of Adam), batch sizes, and increased epoch counts to find the most stable training configuration.

**Declaration**

I, Rushikesh Hande, confirm that the work submitted in this assignment is my own and has been completed following academic integrity guidelines. The code is uploaded on my GitHub repository account, and the repository link is provided below:

GitHub Repository Link: https://github.com/Rushikeshaya/Generative-AI

Signature: Rushikesh Hande

**Submission Checklist**

✔ Research paper details and summary

✔ Code file (Python Notebook or Script)

✔ Dataset or link to the dataset

✔ Visualizations (if applicable)

✔ Screenshots of model performance metrics

✔ Readme File

✔ Comparison with research paper results